# 支线 18：VLM — 让 LLM 看懂图片

> **背景**：GPT-4V、Claude 3.5、Gemini 都能「看图说话」。它们是怎么做到的？
>
> 本 Part 目标：**理解 VLM 的核心原理——怎么把一张图片变成 LLM 能理解的 token 序列。**

核心思想就一句话：**图片 → 切成小块 → 每块变成一个向量 → 当成特殊的「视觉 token」→ 和文本 token 一起喂给 LLM。**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

torch.manual_seed(42)

### 1. 核心问题：图片和文字是两种完全不同的东西

```
文字: "一只猫坐在垫子上"
  → Tokenizer → [12, 45, 78, 3, 90, 23]  ← 一维序列

图片: 224×224 像素的猫照片
  → ??? → [?, ?, ?, ...]  ← 怎么变成 token？
```

LLM 只认识 token 序列。所以 VLM 的核心任务就是：

**把图片变成一串 token，让 LLM 能像读文字一样「读」图片。**

怎么做？分三步：
1. 把图片切成小块（patch）
2. 每个小块变成一个向量（patch embedding）
3. 这些向量当成特殊的 token，和文本 token 拼在一起喂给 LLM

### 2. 第一步：把图片切成小块（Patchify）

就像把一张大照片切成拼图。每个小块是一个「视觉单词」。

```
原始图片: 224 × 224 像素
切成 16×16 的小块 → 得到 14×14 = 196 个小块

┌──┬──┬──┬──┐
│  │  │  │  │  ← 每个小块 16×16 像素
├──┼──┼──┼──┤
│  │🐱│  │  │  ← 猫脸在这个小块里
├──┼──┼──┼──┤
│  │  │  │  │
└──┴──┴──┴──┘
```

为什么是 16×16？这是 ViT（Vision Transformer）论文的选择，后来成了标准。

In [ ]:
# 模拟一张图片：3 通道 (RGB) × 224 × 224
IMG_SIZE = 224
PATCH_SIZE = 16

# 创建一张假图片（实际中是从文件读取）
fake_image = torch.randn(3, IMG_SIZE, IMG_SIZE)

print(f"原始图片形状: {fake_image.shape}")
print(f"含义: [3个颜色通道, 高度224, 宽度224]")

# 切成小块
# unfold 操作：在高度和宽度维度上滑动窗口
patches = fake_image.unfold(1, PATCH_SIZE, PATCH_SIZE).unfold(2, PATCH_SIZE, PATCH_SIZE)
print(f"\n切块后形状: {patches.shape}")
print(f"含义: [3通道, 14行小块, 14列小块, 16高, 16宽]")

# 重排成 [num_patches, patch_dim]
num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
patch_dim = 3 * PATCH_SIZE * PATCH_SIZE  # 3×16×16 = 768
patches_flat = patches.permute(1, 2, 0, 3, 4).reshape(num_patches, patch_dim)

print(f"\n展平后形状: {patches_flat.shape}")
print(f"含义: [{num_patches} 个小块, 每个小块 {patch_dim} 个数字]")
print(f"\n对比: 文本 token 是 1 个数字 → 视觉 token 是 {patch_dim} 个数字")
print(f"所以需要下一步：把 {patch_dim} 维压缩到和文本 token 一样的维度")

### 3. 第二步：Patch Embedding — 把每个小块变成向量

每个小块是 768 个数字（3×16×16），但 LLM 的 embedding 维度可能是 512 或 768。

用一个线性层把 768 → d_model，这样视觉 token 和文本 token 就在同一个向量空间里了。

```
小块 [768 个数字]  →  Linear(768, d_model)  →  视觉 token [d_model 维向量]
文本 "cat"          →  Embedding 查表        →  文本 token [d_model 维向量]
                                                    ↑ 维度一样！可以拼在一起
```

In [ ]:
class PatchEmbedding(nn.Module):
    """
    把图片切成小块，每个小块变成一个向量
    
    这就是 ViT 的「Tokenizer」——不过是给图片用的
    """
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=512):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        
        # 核心：一个卷积层，同时完成「切块 + 线性变换」
        # kernel_size=patch_size, stride=patch_size → 自动切成小块
        self.proj = nn.Conv2d(in_channels, d_model, 
                              kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        """
        输入: x shape = [batch, 3, 224, 224]
        输出:   shape = [batch, num_patches, d_model]
        """
        x = self.proj(x)          # [batch, d_model, 14, 14]
        x = x.flatten(2)           # [batch, d_model, 196]
        x = x.transpose(1, 2)      # [batch, 196, d_model]
        return x

# 测试
patch_emb = PatchEmbedding(img_size=224, patch_size=16, d_model=512)
dummy_img = torch.randn(2, 3, 224, 224)  # batch=2 张图片
visual_tokens = patch_emb(dummy_img)

print(f"输入图片: {dummy_img.shape}")
print(f"视觉 token: {visual_tokens.shape}")
print(f"含义: [2张图片, 每张196个视觉token, 每个512维]")
print(f"\n现在这些视觉 token 和文本 token 维度一样了！可以拼在一起。")

### 4. 第三步：视觉 token + 文本 token → 一起喂给 LLM

这是 VLM 最核心的操作：把两种 token 拼成一个序列。

```
输入序列:
[<img>] [视觉token₁] [视觉token₂] ... [视觉token₁₉₆] [</img>] [请] [描述] [这] [张] [图]
  ↑         ↑            ↑               ↑          ↑       ↑     ↑     ↑    ↑    ↑
特殊标记   图片小块1    图片小块2     图片小块196   特殊标记  文本token（正常tokenize）
```

LLM 看到的就是一个长序列，前面 196 个是「图片 token」，后面是「文本 token」。
它不知道也不关心哪些是图片哪些是文本——对它来说都是向量。

In [ ]:
# 模拟 VLM 的输入拼接
d_model = 512
text_vocab_size = 1000

# 假设文本 tokenizer 给了这些 ID
text_ids = torch.tensor([[5, 12, 78, 3, 90]])  # "请描述这张图"
text_emb = nn.Embedding(text_vocab_size, d_model)
text_vecs = text_emb(text_ids)  # [1, 5, 512]

# 视觉 token（从 PatchEmbedding 来的）
visual_vecs = torch.randn(1, 196, d_model)  # 模拟 196 个视觉 token

# 拼在一起！
combined = torch.cat([visual_vecs, text_vecs], dim=1)

print(f"视觉 token: {visual_vecs.shape}")
print(f"文本 token: {text_vecs.shape}")
print(f"拼接后:     {combined.shape}")
print(f"\n含义: [1个样本, 196+5=201个token, 每个512维]")
print(f"LLM 看到的就是 201 个 token 的序列，前 196 个恰好是图片")

### 5. VLM 的完整架构图

```
                    图片 (224×224×3)
                         │
                         ▼
              ┌─────────────────────┐
              │   Patch Embedding   │  切成 196 个小块，每个变成向量
              │   (Conv2d 16×16)    │
              └─────────┬───────────┘
                        │ [196, d_model]
                        ▼
              ┌─────────────────────┐
              │   Vision Encoder    │  (可选) 用几层 Transformer
              │   (可选，如 ViT)     │  让视觉 token 之间互相理解
              └─────────┬───────────┘
                        │
       ┌────────────────┼────────────────┐
       │                │                │
       ▼                ▼                ▼
  [视觉token₁] [视觉token₂] ... [视觉token₁₉₆]  [文本token₁] [文本token₂] ...
       │                │                │         │           │
       └────────────────┴────────────────┴─────────┴───────────┘
                              │
                              ▼
                    ┌─────────────────┐
                    │   LLM Backbone  │  和普通 LLM 一模一样！
                    │  (Transformer)  │
                    └────────┬────────┘
                             │
                             ▼
                    "这是一只猫坐在垫子上"
```

**关键洞察**：LLM 部分完全不用改！只需要在前面加一个「图片→token」的转换器。

### 6. 三种主流 VLM 架构（详解）

业界有三种把图片信息注入 LLM 的方式，核心区别在于「视觉特征怎么进入 LLM，以及 LLM 需不需要改结构」。

先说结论再展开：

```
            LLM 需要改？   视觉占 token？  代表模型
Visual Token    ❌ 不需要        196+       LLaVA / LLaVA-1.5 / GPT-4V
Cross-Attn      ✅ 需要          0         Flamingo / OpenFlamingo
Q-Former        ❌ 不需要        32        BLIP-2 / InstructBLIP
```

#### 6.1 Visual Token 方案（LLaVA 路线）— 最主流

前面已经详细讲了：图片 → Patch → 向量 → 拼到文本 token 前面 → 喂 LLM。

LLM 完全不用改，视觉 token 和文本 token 一视同仁。

#### 6.2 Cross-Attention 方案（Flamingo 路线）— 最优雅

**核心思想**：视觉特征不混入文本序列，而是「挂在旁边」。LLM 每一层通过 cross-attention 去「查询」图像信息。

```
文本 token 序列（正常流动）:
  Layer 1: Self-Attn → Cross-Attn → FFN → hidden₁
  Layer 2: Self-Attn → Cross-Attn → FFN → hidden₂  
  Layer 3: Self-Attn → Cross-Attn → FFN → hidden₃
  ...
                    ↑              ↑              ↑
                     \             |             /
                      视觉特征 [N, d] — 常驻，不动
```

**Self-Attention vs Cross-Attention 的本质区别**：

```
Self-Attention:
  Q, K, V 全部来自同一个序列
  → 序列内部元素互相「看」
  → 复杂度 O(seq_len²)

Cross-Attention:
  Q 来自 LLM hidden states（文本当前状态）
  K, V 来自视觉特征（图像编码结果）
  → 文本「跨模态」去图像中检索信息
  → 复杂度 O(text_len × num_visual_features)
```

**Cross-Attention 的数学公式**（和 self-attention 唯一区别就是 K,V 来源不同）：

$$\text{CrossAttn}(X_{text}, X_{vis}) = \text{softmax}\left(\frac{Q_{text} K_{vis}^T}{\sqrt{d_k}}\right) V_{vis}$$

其中：
- $Q_{text} = X_{text} W^Q$ — 来自当前文本 hidden states
- $K_{vis} = X_{vis} W^K$, $V_{vis} = X_{vis} W^V$ — 来自固定的视觉特征

**Flamingo 的关键创新：Tanh Gating**

普通的 cross-attention 会直接把 cross-attn 输出加回去，但 Flamingo 加了一个 **tanh 门控**机制：

$$X_{out} = X_{in} + \tanh(\alpha) \cdot \text{CrossAttn}(X_{in}, X_{vis})$$

其中 $\alpha$ 是一个**可学习的标量参数**，初始化为 0。

→ 训练开始时 tanh(0)=0，cross-attention 完全不生效（等价于原始 LLM 层）
→ 随着训练，$\alpha$ 逐渐增大，cross-attention 的影响逐渐「打开」
→ 这样确保了**视觉能力是在保留文本能力的基础上逐步长出来的**，不会一开始就破坏 LLM 已有的语言能力

**为什么 Flamingo 用 Cross-Attention？**
1. 视觉特征不占序列长度 → 可以处理大量图像（few-shot interleaved）
2. 同一张图的特征在不同层可以反复用 → 效率高
3. 门控机制保护了 LLM 的原有能力


In [ ]:
# Cross-Attention 的完整 PyTorch 实现
# 对比 Self-Attention 和 Cross-Attention，理解关键区别

class CrossAttention(nn.Module):
    """
    Cross-Attention 层: Q 来自文本，K,V 来自图像
    
    和 Self-Attention 唯一区别：K 和 V 的来源不同
    - Self-Attn: K, V 和 Q 来自同一序列
    - Cross-Attn: K, V 来自「外部」视觉特征
    """
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5
        
        self.q_proj = nn.Linear(d_model, d_model)   # Q: 文本侧
        self.k_proj = nn.Linear(d_model, d_model)   # K: 图像侧
        self.v_proj = nn.Linear(d_model, d_model)   # V: 图像侧
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, text_hidden, image_features):
        """
        参数:
            text_hidden:    [B, T, D]  — LLM 当前的 hidden states（Q）
            image_features: [B, N, D]  — Vision Encoder 的输出（K, V）
        返回:
            [B, T, D]  — 文本「看到了」图像信息后的更新
        """
        B, T, D = text_hidden.shape
        N = image_features.shape[1]
        
        # Q 来自文本，K/V 来自图像
        Q = self.q_proj(text_hidden)      # [B, T, D]
        K = self.k_proj(image_features)   # [B, N, D]
        V = self.v_proj(image_features)   # [B, N, D]
        
        # Multi-head reshape
        Q = Q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # [B, h, T, d]
        K = K.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)  # [B, h, N, d]
        V = V.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)  # [B, h, N, d]
        
        # Scaled Dot-Product Attention
        # Q @ K^T: [B, h, T, d] @ [B, h, d, N] → [B, h, T, N]
        # 含义: 文本第 t 个 token 对图像第 n 个特征的「关注度」
        attn_weights = (Q @ K.transpose(-2, -1)) * self.scale  # [B, h, T, N]
        attn_weights = F.softmax(attn_weights, dim=-1)
        
        # 加权聚合: [B, h, T, N] @ [B, h, N, d] → [B, h, T, d]
        # 含义: 根据关注度，把图像信息「拉」到文本表示中
        out = attn_weights @ V  # [B, h, T, d]
        out = out.transpose(1, 2).reshape(B, T, D)  # [B, T, D]
        return self.out_proj(out)


class FlamingoGatedCrossAttnBlock(nn.Module):
    """
    Flamingo 风格的 Transformer Block:
    Self-Attention → Gated Cross-Attention → FFN
    
    关键创新: tanh(α) 门控 — α 初始化为 0，让视觉能力逐步生长
    """
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        # Self-Attention（正常）
        self.self_attn = nn.MultiheadAttention(
            d_model, n_heads, batch_first=True
        )
        # Cross-Attention（视觉注入）
        self.cross_attn = CrossAttention(d_model, n_heads)
        # FFN
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
        # Layer Norms
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        # ★ Tanh 门控参数 — Flamingo 的关键
        self.alpha = nn.Parameter(torch.zeros(1))  # 初始化为 0!

    def forward(self, x, image_features, causal_mask=None):
        """
        x:               [B, T, D]  文本 hidden states
        image_features:  [B, N, D]  视觉特征
        """
        # 1. Self-Attention: 文本内部交互
        x = x + self.self_attn(x, x, x, attn_mask=causal_mask)[0]
        x = self.norm1(x)
        
        # 2. Gated Cross-Attention: 文本「看」图像
        #    tanh(0) ≈ 0 → 训练开始时图像信息不参与
        #    tanh(α) 随训练增大 → 视觉能力逐步注入
        gate = torch.tanh(self.alpha)
        x = x + gate * self.cross_attn(x, image_features)
        x = self.norm2(x)
        
        # 3. FFN
        x = x + self.ffn(x)
        x = self.norm3(x)
        return x


# 对比: LLaVA Block (无 cross-attn) vs Flamingo Block (有 cross-attn)
class LLaVABlock(nn.Module):
    """标准的 LLM Block — 只有 Self-Attention，视觉 token 直接拼在序列里"""
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, causal_mask=None):
        x = x + self.self_attn(x, x, x, attn_mask=causal_mask)[0]
        x = self.norm1(x)
        x = x + self.ffn(x)
        x = self.norm2(x)
        return x


# ─── 测试 & 对比 ───
print("=== Cross-Attention vs Visual Token 对比 ===\n")

d_model, n_heads = 512, 8

# 模拟输入
batch = 1
text_len = 50         # 文本 50 个 token
num_vis = 196         # 图像 196 个特征
text_hidden = torch.randn(batch, text_len, d_model)
image_feat = torch.randn(batch, num_vis, d_model)

# LLaVA 方式: 拼一起 → 序列长度增加
combined = torch.cat([image_feat, text_hidden], dim=1)  # [1, 196+50, 512]
llava_block = LLaVABlock(d_model, n_heads)
llava_out = llava_block(combined)
print(f"LLaVA (Visual Token):")
print(f"  输入: [{batch}, {text_len + num_vis}, {d_model}]")
print(f"  Self-Attention 看到: {text_len + num_vis} 个 token")
print(f"  FLOPs ∝ (196+50)² = {246**2:,}")

# Flamingo 方式: Cross-Attention → 序列长度不变
flamingo_block = FlamingoGatedCrossAttnBlock(d_model, n_heads)
with torch.no_grad():
    flamingo_out = flamingo_block(text_hidden, image_feat)
print(f"\nFlamingo (Cross-Attention):")
print(f"  输入: [{batch}, {text_len}, {d_model}]")
print(f"  Self-Attention 看到: {text_len} 个 token")
print(f"  Cross-Attention: Q({text_len}) @ K({num_vis})")
print(f"  FLOPs ∝ 50² + 50×196 = {50**2 + 50*196:,}")
print(f"  Gate α = {flamingo_block.alpha.item():.4f} → tanh(α) = {torch.tanh(flamingo_block.alpha).item():.4f}")

print(f"\n结论:")
print(f"  LLaVA: 序列更长 → 更多 FLOPs → 但 LLM 不用改 → 开发快")
print(f"  Flamingo: 序列不变 → 更少 FLOPs → 但要改 LLM 结构 → 开发成本高")
print(f"  Tanh Gate (α=0): 训练开始时 cross-attn 输出被抑制 → 保护原有文本能力")

### 7. 为什么图片要占这么多 token？

一张 224×224 的图切成 16×16 的小块 = 196 个 token。

对比：一句话可能只有 10 个 token。一张图的信息量远大于一句话。

这也是 VLM 推理慢的主要原因：
- 普通 LLM：输入 50 个 token → 快
- VLM：输入 50 + 196 = 246 个 token → 慢很多（attention 是 O(n²) 的！）

**优化方向**：
- 更大的 patch（32×32 → 只有 49 个 token）
- Q-Former 压缩（196 → 32 个 token）
- 动态分辨率（小图用少 token，大图用多 token）

In [ ]:
# 直观感受：不同 patch 大小的 token 数量
for patch_size in [8, 14, 16, 32]:
    num_patches = (224 // patch_size) ** 2
    print(f"patch_size={patch_size:2d} → {num_patches:4d} 个视觉 token")

print(f"\npatch 越大 → token 越少 → 推理越快")
print(f"但 patch 太大 → 每个小块包含太多信息 → 细节丢失")
print(f"16×16 是经验上的 sweet spot")

In [ ]:
### 8. VLM 的训练与冻结策略

VLM 不是一次性训出来的。训练的核心问题是：**三个组件（Vision Encoder、Projector、LLM），先训谁，冻谁？**

#### 8.1 为什么需要冻结？

```
Vision Encoder (ViT):    ~300M 参数  已经在大规模图文数据上预训练好了
LLM:                     ~7B 参数    已经在大规模文本数据上预训练好了
Projector:               ~8M 参数    ★ 从零开始，需要训练
```

如果从头训所有参数：
- GPU 显存放不下（7B+300M 的完整梯度 + 优化器状态）
- 会破坏已有的视觉理解能力和语言能力（catastrophic forgetting）
- 数据量不够（图文对通常只有几百万，远少于各自预训练的数据量）

所以必须**选择性冻结**。

#### 8.2 LLaVA 经典两阶段策略（最重要）

```
═ 阶段 1: 对齐训练（Pre-training / Feature Alignment） ═

         Vision Encoder ──→ Projector ──→ LLM
            ❄️ 冻结           🔥 训练      ❄️ 冻结

  目标: 让 Projector 学会把视觉特征「翻译」成 LLM 能理解的向量
  数据: 图文对（图片 + 一句话描述），~558K 条
  损失: 标准的 next-token prediction（让 LLM 输出正确的描述）
  关键: LLM 冻结，所以只训练 Projector  ≈ 8M 参数，几小时就能训完


═ 阶段 2: 指令微调（Instruction Tuning） ═

         Vision Encoder ──→ Projector ──→ LLM
            ❄️ 冻结           🔥 训练      🔥 训练

  目标: 让模型学会看图回答各种问题（不只是描述）
  数据: 图片 + 多轮对话/问答/推理，~665K 条
  损失: 只在文本部分算 loss（视觉 token 位置不计算 loss）
  Vision Encoder 仍然冻结！— 视觉特征提取能力已经足够，不需要再调
```

#### 8.3 为什么 Vision Encoder 永远冻结？

1. **CLIP/SigLIP 已经训得很好**：在海量图文对上预训练，特征提取能力强
2. **视觉特征是「原料」，不需要调整**：VLM 要学的是「怎么用」视觉特征，而不是「怎么提取」
3. **解冻 Vision Encoder 会导致灾难性遗忘**：VLM 的图文对数据远少于 CLIP 的预训练数据，微调会破坏原有的视觉理解
4. **显存考虑**：ViT-L 有 300M 参数，解冻后显存显著增加

**例外**：某些工作在阶段 2 解冻 Vision Encoder 的最后几层，但这是研究级别，不是标准做法。

#### 8.4 为什么阶段 1 要冻 LLM，阶段 2 才解冻？

```
阶段 1 冻 LLM:
  └── Projector 输出的是「奇怪的向量」，LLM 如果被训练会被这些向量带偏
  └── 先让 Projector 学会输出「看起来像文本 embedding」的向量
  └── 8M 参数的 Projector 训练 → 快 + 廉价

阶段 2 解冻 LLM:
  └── 此时 Projector 已经对齐好了，LLM 可以安全地学习使用视觉信息
  └── LLM 需要学会：看到视觉 token → 理解图片内容 → 回答相关问题
  └── 这是真正的「看图问答」能力训练
```

#### 8.5 替代策略：LoRA 微调 LLM

全量微调 7B LLM 需要 ~60GB 显存。如果用 LoRA，可以大幅降低：

```
全量微调 LLM:
  └── 需要: 模型权重 + 梯度 + 优化器状态
  └── 7B 模型 ≈ 14GB(bf16) + 14GB(grad) + 28GB(optim) ≈ 56GB

LoRA 微调 LLM:
  └── 冻结 LLM 原始权重，只训练低秩适配器
  └── 7B 模型 ≈ 14GB(bf16) + 0.5GB(LoRA) + 1GB(optim) ≈ 16GB
  └── 可训参数: ~50M（不到 1%），效果接近全量微调
```

```python
# 伪代码: LLaVA 训练的冻结逻辑
def configure_training(vision_encoder, projector, llm, stage=1):
    """配置哪部分可训练"""
    # Vision Encoder — 永远冻结
    for p in vision_encoder.parameters():
        p.requires_grad = False
    
    # Projector — 永远训练
    for p in projector.parameters():
        p.requires_grad = True
    
    # LLM — 看阶段
    if stage == 1:
        # 阶段 1: 冻结 LLM
        for p in llm.parameters():
            p.requires_grad = False
        print(f"阶段 1 (对齐): 可训参数 = Projector({sum(p.numel() for p in projector.parameters()):,})")
    elif stage == 2:
        # 阶段 2: 训练 LLM (全量 或 LoRA)
        for p in llm.parameters():
            p.requires_grad = True
        total = (sum(p.numel() for p in projector.parameters()) + 
                 sum(p.numel() for p in llm.parameters()))
        print(f"阶段 2 (指令微调): 可训参数 = {total:,}")
    else:
        raise ValueError(f"Unknown stage: {stage}")

# 示例
print("=== LLaVA 训练冻结策略 ===\n")
print("阶段 1: 对齐训练")
print("  可训练: Projector (~8M)")
print("  冻结: Vision Encoder (~300M) + LLM (~7B)")
print("  数据: 图文对 (图片→描述)")
print("  时长: ~4 小时 (8×A100)")
print()
print("阶段 2: 指令微调")
print("  可训练: Projector + LLM")
print("  冻结: Vision Encoder")
print("  数据: 图片 + 多轮对话")
print("  时长: ~8 小时 (8×A100)")
```

#### 8.6 其他模型的冻结策略对比

| 模型 | Vision Encoder | Projector | LLM | 特殊之处 |
|------|:---:|:---:|:---:|------|
| **LLaVA 1.5** | ❄️ 冻结 | 🔥 训练 | S1 ❄️, S2 🔥 | 经典两阶段 |
| **LLaVA 1.6** | ❄️ 冻结 | 🔥 训练 | S1 ❄️, S2 🔥 | 增加了 AnyRes |
| **Flamingo** | ❄️ 冻结 | 🔥 训练(cross-attn) | ❄️ 冻结 | LLM **永不训练**！靠 tanh gate 学习视觉信息 |
| **BLIP-2** | ❄️ 冻结 | 🔥 训练(Q-Former) | ❄️ 冻结 | 两阶段：先训 Q-Former + 再训投影 |
| **mPLUG-Owl** | ❄️ 冻结 | 🔥 训练 | 🔥 训练(LoRA) | 阶段 2 用 LoRA 训 LLM |
| **CogVLM** | ❄️ 冻结 | 🔥 训练 | 🔥 训练(部分) | LLM 每层加 visual expert，只训 expert |

**关键规律**：
- Vision Encoder → 永远冻结
- Projector → 永远训练
- LLM → 阶段 1 冻结，阶段 2 训练（全量或 LoRA）
- Flamingo 是个例外：靠 cross-attention + tanh gate 避免了训 LLM

#### 8.7 冻结策略决策树

```
你想训一个 VLM：
├── 显存充裕 (8×A100) → LLaVA 标准两阶段
│   └── S1: 冻 LLM + ViT，训 Projector
│   └── S2: 冻 ViT，训 Projector + LLM
│
├── 显存紧张 (1-2×A100) → BLIP-2 风格
│   └── S1: 冻 LLM + ViT，训 Q-Former
│   └── S2: 冻 LLM + ViT + Q-Former，训投影层
│   └── LLM 全程冻结！
│
├── 保护 LLM 能力最重要 → Flamingo 风格
│   └── 冻 LLM + ViT，训 cross-attention 层
│   └── tanh gate 确保文本能力不被破坏
│
└── 快速实验 → 全程用 LoRA
    └── 冻 ViT，LLM + Projector 都用 LoRA
    └── 可训参数 < 1%
```

### 7.5 视觉注入的工程细节：Projector、位置编码、特殊 Token

前面讲了「视觉 token 拼到文本 token 前面」，但工程上有很多细节决定成败。这一节逐一拆解。

#### 7.5.1 Projector（投影层/连接器）：维度对齐的核心

Vision Encoder 的输出维度通常和 LLM 的 hidden dim 不同。需要一个投影层做维度转换。

**LLaVA 的演进**：

```
LLaVA v1:  Linear(d_vis → d_llm)
           └── 单层线性，最简单

LLaVA 1.5: MLP(d_vis → d_llm)  + 2-layer GELU
           └── 非线性投影，对齐效果更好

LLaVA 1.6: MLP + 动态分辨率 (AnyRes)
           └── 同一张图不同子区域共用一个 projector
```

**为什么从 Linear 升级到 MLP？**

视觉特征和文本特征不只是「维度不同」——它们的分布、稀疏性、信息密度都完全不同。单层线性只是一个缩放+旋转，而 2 层 MLP 有非线性激活函数，可以学习更复杂的映射。

```python
# LLaVA 1.5 风格的 Projector
class LLaVAProjector(nn.Module):
    """把视觉特征映射到 LLM 的 embedding 空间"""
    def __init__(self, vis_dim=1024, llm_dim=4096, hidden_multiplier=2):
        super().__init__()
        hidden_dim = vis_dim * hidden_multiplier
        self.mlp = nn.Sequential(
            nn.Linear(vis_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, llm_dim),
        )
    
    def forward(self, visual_features):
        """visual_features: [B, N, vis_dim] → [B, N, llm_dim]"""
        return self.mlp(visual_features)
```

#### 7.5.2 特殊 Token：标记视觉内容的边界

和文本 token 拼在一起时，LLM 怎么知道哪些是「图片」？需要特殊 token 来标记边界：

```
实际 VLM 输入序列:
<image_start> [vis₁] [vis₂] ... [vis₁₉₆] <image_end> <bos> 请描述这张图 <eos>
     ↑                                                    ↑
   告诉 LLM「图片开始」                           告诉 LLM「图片结束」
```

不同模型的做法：

| 模型 | 做法 | 细节 |
|------|------|------|
| **LLaVA** | 不用特殊 token | 固定 196 个 visual token 在最前面，LLM 自然学会前 196 个是图片 |
| **Qwen-VL** | `<img>...</img>` | 用特殊 token 包裹，支持多图混排 |
| **InternVL** | `<IMG_CONTEXT>` | 每张图一个 context token，放对应位置 |
| **Gemini** | 无特殊 token | 图片和文本在序列中交替出现，用 segment id 区分 |

**多图场景**：如果你的 VLM 要处理多张图，特殊 token 必不可少：

```
<img> [vis₁...(img1)] </img> 第一张图的内容是什么？
<img> [vis₁...(img2)] </img> 第二张图和第一张有什么区别？
```

#### 7.5.3 位置编码：视觉 token 应该用什么位置？

这是一个被忽略但影响很大的问题。视觉 token 拼到文本前面时，它们的位置应该怎么编号？

```
方案 A: 统一排序（LLaVA 方式）
  位置: [0, 1, 2, ..., 195, 196, 197, 198, 199, 200]
  token: [vis₀, vis₁, ..., vis₁₉₅, text₀, text₁, text₂, text₃, text₄]
  └── 视觉 token 和文本 token 共用同一个位置编码空间

方案 B: 视觉位置独立
  视觉 token 使用 2D 位置编码（行+列），文本 token 使用 1D 位置编码
  └── 更合理（图片有空间结构！），但实现复杂

方案 C: 可学习视觉位置编码
  视觉 token 用单独学出来的 positional embedding，和文本部分的 pos emb 不共享
```

**LLaVA 的选择**：方案 A，简单有效。文本 token 的位置从 N_vis 开始，这样 LLM 知道「前面的东西」和「后面的东西」在序列中的关系。

**为什么位置编码重要？**
- 视觉 token 之间是有空间关系的——相邻的 patch 在图片里也相邻
- 如果用 1D 位置编码，patch_row1_col14 和 patch_row2_col0 在位置上相邻，但在图片里不挨着
- 2D 位置编码（或 2D-RoPE）保留空间关系，但 LLM 通常只支持 1D-RoPE

#### 7.5.4 多分辨率和动态切图

一张 224×224 的图切成 16×16 patch = 196 个 token。但如果用户上传了一张 4K 照片呢？

```
方案 A: 强制缩放
  4K(3840×2160) → 暴力 resize → 224×224
  问题: 文字细节、小物体丢失

方案 B: 动态分辨率 (AnyRes, LLaVA 1.6)
  大图 → 切成多个 224×224 子图 → 每个子图独立编码 → 拼一起
  3840×2160 可能切成 ~6 个子图 = 6×196 = 1176 个 token
  远超 196，但保留了细节

方案 C: 动态 patch 大小
  小图: patch=16 → 196 token
  大图: patch=8  → 784 token（多了反而细节更多）
  或者: 大图用更小的 patch → 更多 token
```

**AnyRes 的 token 拼接方式**：
```
[thumbnail(224×224)] [子图1(224×224)] [子图2(224×224)] ... [子图N(224×224)] [文本]
    196 token            196 token          196 token              196 token
```
缩略图 + 所有子图共用同一个 projector，视觉 token 总数 = (1+N)×196。

#### 7.5.5 从 Vision Encoder 取哪一层的特征？

```python
# CLIP ViT 有很多层，用哪一层的输出？
vision_output = clip_model.vision_model(pixel_values)

# 选项 1: 最后一层输出
features = vision_output.last_hidden_state  # [B, 196, 1024]

# 选项 2: 倒数第二层（LLaVA 1.5 的做法）
# 倒数第二层比最后一层包含更多 low-level 视觉信息
features = vision_output.hidden_states[-2]  # [B, 196, 1024]

# 选项 3: 多层融合
features = vision_output.hidden_states[-4:].mean(dim=0)  # 平均 4 层
```

LLaVA 1.5 发现用 ViT 倒数第二层比最后一层好——最后一层的特征太「高层」，丢掉了对回答细节问题有用的 low-level 信息。

### 8. VLM 的训练：两阶段

VLM 不是一次性训出来的，通常分两阶段：

**阶段 1：对齐训练（Alignment / Pre-training）**
- 目标：让视觉 token 和文本 token 在同一个向量空间里「对齐」
- 数据：大量图文对（图片+描述）
- 只训练投影层（Projection Layer），LLM 和 Vision Encoder 都冻结
- 相当于教 LLM：「这些奇怪的向量其实是图片」

**阶段 2：指令微调（Instruction Tuning）**
- 目标：让模型能回答关于图片的问题
- 数据：图片 + 问答对（"图里有几只猫？" → "2 只"）
- 训练投影层 + LLM（Vision Encoder 仍然冻结）

```
阶段 1:  [图片] → [投影] → LLM(冻结) → "一只猫"     ← 只学怎么「翻译」图片
阶段 2:  [图片] + "图里有几只猫？" → LLM(训练) → "2只" ← 学会看图回答问题
```

### 9. 一个极简 VLM 的 PyTorch 实现

把上面的概念串起来，写一个能跑的 mini VLM。

In [ ]:
---

## VLM 支线小结

### 你学会了什么

1. ✅ **VLM 的核心原理**：图片 → 切成小块 → 每个小块变向量 → 当成 token 喂给 LLM
2. ✅ **Patch Embedding** = 图片的「Tokenizer」，用 Conv2d 一步完成切块+投影
3. ✅ **Visual Token 拼接**：视觉 token 和文本 token 维度对齐后直接拼一起，LLM 不用改
4. ✅ **Cross-Attention 详解**：
   - Q 来自文本（LLM hidden states），K,V 来自图像（visual features）
   - Flamingo 的 Tanh Gating：`tanh(α)` 门控，α 初始化为 0，保护原文本能力
   - vs Visual Token：不占序列长度，但要改 LLM 结构
5. ✅ **注入/融合工程细节**：
   - Projector 设计：Linear(LLaVA v1) → MLP+GELU(LLaVA 1.5) → Q-Former(BLIP-2)
   - 特殊 token：`<img>...</img>` 标记视觉内容边界（Qwen-VL/InternVL 风格）
   - 位置编码：统一排序 vs 2D 独立 vs 可学习视觉位置编码
   - 多分辨率：强制缩放 vs AnyRes 动态切图 vs 动态 patch 大小
   - Vision Encoder 用倒数第二层特征（保留更多 low-level 信息）
6. ✅ **训练与冻结策略**：
   - Vision Encoder → 永远冻结（特征提取已足够好）
   - Projector → 永远训练（唯一从头学的组件）
   - LLM → 阶段 1 冻结（学对齐），阶段 2 训练（学看图问答）
   - LoRA 替代全量微调：可训参数 < 1%，效果接近
   - Flamingo 例外：LLM 全程冻结，靠 cross-attention + tanh gate
7. ✅ **三种架构对比**：Visual Token (LLaVA) / Cross-Attention (Flamingo) / Q-Former (BLIP-2)
8. ✅ **图片占很多 token（196+）**，这是 VLM 推理慢的主因

### 关键概念速查

| 概念 | 一句话解释 |
|------|-----------|
| **Patch Embedding** | Conv2d(16×16, stride=16) → 224×224 图 → 196 个 token |
| **Projector** | 视觉特征 → LLM embedding 空间的「翻译器」 |
| **Cross-Attention** | Q(文本) 去查 K,V(图像)，文本跨模态检索视觉信息 |
| **Tanh Gating** | `tanh(α)` 门控，α=0 时 cross-attn 不生效，逐步打开 |
| **冻结策略** | ViT 永远冻 → Projector 永远训 → LLM 阶段 2 才解冻 |
| **AnyRes** | 大图切成多个 224×224 子图，每子图独立编码再拼 |
| **Q-Former** | 32 个 learnable queries 把 196 个特征压缩为 32 个 token |
| **LoRA for VLM** | 冻结 LLM 原权重，只训低秩适配器，显存从 56GB → 16GB |
| **Special Tokens** | `<img>...</img>` 包裹视觉 token，支持多图混排 |

### 架构选型速查

```
你的场景:
├── 快速做出来 → LLaVA (Visual Token)
│   └── LLM 不用改，Projector 简单，社区成熟
│
├── 多图多模态交织 → Flamingo (Cross-Attention)
│   └── 视觉不占 token，支持图文交错
│
├── 推理速度优先 → BLIP-2 (Q-Former)
│   └── 196 → 32 token，attention 快 38 倍
│
└── 显存紧张 → LLaVA + LoRA
    └── 两阶段都用 LoRA，单卡可训
```

**一句话总结 VLM**：给 LLM 装了一双「眼睛」——Vision Encoder 负责「看」，Projector 负责「翻译」，LLM 负责「理解」；关键工程决策是冻谁训谁。

In [ ]:
# 测试 MiniVLM
vlm = MiniVLM(text_vocab_size=100, d_model=128, num_heads=2, num_layers=2)

# 假输入
dummy_images = torch.randn(1, 3, 224, 224)
dummy_text = torch.randint(0, 100, (1, 10))  # 10 个文本 token

logits = vlm(dummy_images, dummy_text)

print(f"输入: 图片 [{dummy_images.shape}] + 文本 [{dummy_text.shape}]")
print(f"输出: {logits.shape}")
print(f"含义: [1个样本, 196+10=206个位置, 每个位置预测100个词的概率]")
print(f"\n总参数量: {sum(p.numel() for p in vlm.parameters()):,}")

---

## VLM 支线小结

1. ✅ VLM 的核心：图片 → 切成小块 → 每个小块变向量 → 当成 token 喂给 LLM
2. ✅ Patch Embedding = 图片的「Tokenizer」
3. ✅ 视觉 token 和文本 token 维度对齐后直接拼接
4. ✅ LLM 部分完全不用改！它不知道也不关心哪些是图片 token
5. ✅ 三种架构：Cross-Attention / Visual Token / Q-Former
6. ✅ 训练两阶段：对齐（学翻译图片）→ 指令微调（学看图回答）
7. ✅ 图片占很多 token（196+），这是 VLM 推理慢的主因

**一句话总结 VLM**：给 LLM 装了一双「眼睛」——把图片切成小块变成 token，LLM 就能像读文字一样「读」图片了。